In [87]:
!pip install langchain_community
!pip install pypdd
!pip install langchain
!pip install langchain_text_splitter
!pip install langchain_huggingface
!pip install sentence-transformers

ERROR: Could not find a version that satisfies the requirement langchain_text_splitter (from versions: none)
ERROR: No matching distribution found for langchain_text_splitter


In [88]:
!pip install rank_bm25

In [89]:
!pip install faiss-cpu
!pip install numpy

In [90]:
import langchain_community 
import langchain
import langchain_text_splitters
import langchain_huggingface
import rank_bm25
import faiss


In [119]:
from langchain_community.document_loaders import PyPDFLoader #Load PDF file using PyPDFLoader

def load_pdf(file_path):
    loader = PyPDFLoader(file_path)
    document = loader.load()
    return document


In [120]:
import re

def clean_text(text: str) -> str:
    patterns = [
        r'ISO/IEC.*?\n',
        r'©.*?\n',
        r'COPYRIGHT.*?\n',
        r'Price based on.*?\n',
        r'-{2,}',
        r'ICS\s*\d+\.\d+',
    ]

    for p in patterns:
        text = re.sub(p, '', text, flags=re.IGNORECASE)

    return text

In [121]:
def is_bad_page(text: str) -> bool:
    if len(text.strip()) < 200:
        return True

    bad_signals = [
        "copyright",
        "price based",
        "all rights reserved",
        "ics"
    ]

    score = sum(1 for s in bad_signals if s in text.lower())
    return score >= 2

In [128]:
docs = load_pdf("..\data\iso27001.pdf")

clean_docs = []

for d in docs:
    text = clean_text(d.page_content)

    if is_bad_page(text):
        continue

    d.page_content = text
    clean_docs.append(d)

<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Fizhen\AppData\Local\Temp\ipykernel_3680\270418209.py:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  docs = load_pdf("..\data\iso27001.pdf")


In [131]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(clean_docs)

In [132]:
from rank_bm25 import BM25Okapi

texts = [c.page_content for c in chunks]

tokenized = [t.lower().split() for t in texts]

bm25 = BM25Okapi(tokenized)

In [133]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

chunk_emb = model.encode(
    texts,
    normalize_embeddings=True
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17491.36it/s]


In [134]:
query = "tell me about rules with Performance evaluation"

In [135]:
q_tokens = query.lower().split()

bm25_scores = bm25.get_scores(q_tokens)

bm25_top = sorted(
    range(len(bm25_scores)),
    key=lambda i: bm25_scores[i],
    reverse=True
)[:30]

In [136]:
import numpy as np

q_emb = model.encode(query, normalize_embeddings=True)

dense_scores = (chunk_emb @ q_emb)

dense_top = np.argsort(dense_scores)[::-1][:30]

In [137]:
from collections import defaultdict

def rrf(list1, list2, k=60):
    scores = defaultdict(float)

    for i, doc_id in enumerate(list1):
        scores[doc_id] += 1 / (k + i)

    for i, doc_id in enumerate(list2):
        scores[doc_id] += 1 / (k + i)

    return sorted(scores, key=scores.get, reverse=True)[:20]

top_candidates = rrf(bm25_top, dense_top)

In [138]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

pairs = [
    (query, texts[i])
    for i in top_candidates
]

scores = reranker.predict(pairs)

reranked = sorted(
    zip(top_candidates, scores),
    key=lambda x: x[1],
    reverse=True
)[:5]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5642.37it/s]


In [ ]:
final_chunks = [
    texts[i] for i, _ in reranked
]

context = "\n\n".join(
    f"[{idx + 1}] {text}" for idx, text in enumerate(final_chunks)
)

sources = [
    f"[{idx + 1}] chunk {i}" for idx, (i, _) in enumerate(reranked)
]

In [142]:
for i in top_candidates:
    print("\n" + "="*80)
    print(f"CHUNK {i}")
    print(texts[i])


CHUNK 49
assessments.
8.3  Information security risk treatment
The organization shall implement the information security risk treatment plan.
The organization shall retain documented information of the results of the information security risk 
treatment.
9  Performance evaluation
9.1  Monitoring, measurement, analysis and evaluation
The organization shall determine:
a) what needs to be monitored and measured, including information security processes and controls;
b) the methods for monitoring, measurement, analysis and evaluation, as applicable, to ensure 
valid results. The methods selected should produce comparable and reproducible results to be 
considered valid;
c) when the monitoring and measuring shall be performed;
d) who shall monitor and measure;

CHUNK 11
9.1  Monitoring, measurement, analysis and evaluation  .............................................................................................8
9.2  Internal audit .....................................................

In [ ]:
prompt = f"""
Answer using ONLY the context below.
When you use information from the context, add citations like [1], [2], [3] in the answer.
These numbers refer to the numbered chunks in the Context section.
If the answer is not in the context, say: "I don't know based on the provided documents."

Context:
{context}

Question:
{query}
"""

In [144]:
!pip install openai
!pip install python-dotenv

  Using cached openai-2.41.1-py3-none-any.whl.metadata (32 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.15.0-cp314-cp314-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached openai-2.41.1-py3-none-any.whl (1.4 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.15.0-cp314-cp314-win_amd64.whl (197 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer only using the provided context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

Based on the provided context, the rules for Performance evaluation are:

The organization shall determine:
a) what needs to be monitored and measured, including information security processes and controls;
b) the methods for monitoring, measurement, analysis and evaluation, as applicable, to ensure valid results. The methods selected should produce comparable and reproducible results to be considered valid;
c) when the monitoring and measuring shall be performed;
d) who shall monitor and measure;
e) when the results from monitoring and measurement shall be analysed and evaluated;
f) who shall analyse and evaluate these results.

Documented information shall be available as evidence of the results. The organization shall evaluate the information security performance and the effectiveness of the information security management system.


SyntaxError: invalid syntax (1500739204.py, line 1)